# 🧪 Reaction SMIRKS Builder & Enumeration Lab

A single, end-to-end notebook to **design, generalize, and test** reaction templates.

**The workflow:**
1. **Draw** your example — paste the reagent(s) and the expected product as SMILES.
2. **Generalize** — turn that single example into a reusable reaction SMARTS/SMIRKS template (automatic or manual).
3. **Validate** — confirm the template re-creates your expected product (round-trip sanity check).
4. **Enumerate** — apply the template across reagent libraries supplied as **`.tsv`** files.
5. **Visualize** — render every reaction in 2D, right here in the notebook.
6. **Export** — optionally save products to CSV / an Excel workbook with embedded reaction images.

> Works for **one-reactant** and **two-reactant** reactions. Just give 1 or 2 reagent SMILES in the config cell.

## 0 · Setup

RDKit is the only hard requirement. Two optional helpers unlock *automatic* template generation:

| Engine | Needs | What it does |
|---|---|---|
| **RDKit** | always | parse, enumerate, draw, validate |
| **rdchiral** | light (pure-python + RDKit) | extract a template from an **atom-mapped** example |
| **rxnmapper + rxnutils** | heavy (PyTorch) | **auto atom-map** an unmapped example, then extract a generalized template |

Run the install cell once if anything is missing, then the imports cell.

In [ ]:
# Run once if needed. RDKit is required; the others are optional (see table above).
# %pip install rdkit
# %pip install rdchiral                                   # lightweight auto-template (mapped input)
# ---- heavy / optional: automatic atom mapping of UNMAPPED examples ----
# %pip install --no-deps reaction-utils
# %pip install "PyYAML>=5.4.1,<6.0.0" "Deprecated>=1.2.13,<2.0.0" wrapt-timeout-decorator xxhash rxnmapper

In [ ]:
import itertools, io, os
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, Descriptors, Crippen, rdChemReactions
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display
RDLogger.DisableLog("rdApp.*")          # silence noisy valence warnings during enumeration

# --- detect optional template engines ---
HAVE_RDCHIRAL = HAVE_RXNMAPPER = False
try:
    from rdchiral.template_extractor import extract_from_reaction
    HAVE_RDCHIRAL = True
except Exception:
    pass
try:
    from rxnmapper import RXNMapper
    from rxnutils.chem.reaction import ChemicalReaction
    HAVE_RXNMAPPER = True
except Exception:
    pass

print("RDKit       : ready")
print("rdchiral    :", "ready" if HAVE_RDCHIRAL else "not installed (manual / rxnmapper paths still work)")
print("rxnmapper   :", "ready" if HAVE_RXNMAPPER else "not installed (manual / rdchiral paths still work)")

## 1 · Toolkit

All reusable functions live here so the workflow cells below stay short and readable.
You normally won't need to edit this cell.

In [ ]:
# ----------------------------- molecules -----------------------------
def to_mol(smiles):
    """SMILES -> sanitized RDKit Mol (or None)."""
    return Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None

def canon(smiles):
    """Canonical SMILES, or None if unparseable."""
    m = to_mol(smiles)
    return Chem.MolToSmiles(m) if m else None

def props(mol):
    """(MW, logP) rounded to 2 dp."""
    return round(Descriptors.MolWt(mol), 2), round(Crippen.MolLogP(mol), 2)

# ----------------------------- enumeration ---------------------------
def run_template(rxn, reactant_mols):
    """Apply a reaction to a tuple of reactant Mols. Returns {canonical_smiles: Mol} (deduplicated)."""
    out = {}
    for product_set in rxn.RunReactants(tuple(reactant_mols)):
        for p in product_set:
            try:
                Chem.SanitizeMol(p)
                out.setdefault(Chem.MolToSmiles(p), p)
            except Exception:
                continue
    return out

def validate_template(smarts, reactant_smiles, expected_product_smiles):
    """Round-trip check: does `smarts` regenerate the expected product from the example reactants?"""
    rxn = rdChemReactions.ReactionFromSmarts(smarts)
    mols = [to_mol(s) for s in reactant_smiles]
    if any(m is None for m in mols):
        return {"ok": False, "reason": "a reactant SMILES failed to parse", "produced": []}
    produced = run_template(rxn, mols)
    expected = canon(expected_product_smiles)
    return {"ok": expected in produced,
            "expected": expected,
            "produced": list(produced.keys()),
            "n_products": len(produced)}

# ----------------------------- drawing -------------------------------
def reaction_png(reactant_mols, product_mol, width=760, height=240, highlight=True):
    """Render reactants >> product to PNG bytes (template-atom highlighting optional)."""
    mols = list(reactant_mols) + [product_mol]
    for m in mols:
        m.UpdatePropertyCache(strict=False)
        Chem.GetSSSR(m)
        AllChem.Compute2DCoords(m)
    rxn = rdChemReactions.ChemicalReaction()
    for rm in reactant_mols:
        rxn.AddReactantTemplate(rm)
    rxn.AddProductTemplate(product_mol)
    d = rdMolDraw2D.MolDraw2DCairo(width, height)
    d.drawOptions().prepareMolsBeforeDrawing = False
    d.DrawReaction(rxn, highlightByReactant=highlight)
    d.FinishDrawing()
    return d.GetDrawingText()

def show_reaction(reactant_mols, product_mol, label="", **kw):
    """Display a single reaction inline."""
    from IPython.display import Image
    if label:
        print(label)
    display(Image(data=reaction_png(reactant_mols, product_mol, **kw)))

def show_template(smarts):
    """Pretty-print + draw a reaction template."""
    print(smarts)
    return rdChemReactions.ReactionFromSmarts(smarts)

# ----------------------------- I/O (.tsv) ----------------------------
def load_library(path, smiles_col=None, id_col=None):
    """
    Load a reagent library from a TSV (or CSV). Auto-detects the SMILES and ID columns.
    Returns a DataFrame with guaranteed columns: smiles, id, mol  (unparseable rows dropped).
    """
    sep = "\t" if str(path).lower().endswith((".tsv", ".tab")) else None  # None => pandas sniffs
    df = pd.read_csv(path, sep=sep, engine="python")
    if smiles_col is None:
        smiles_col = next((c for c in df.columns if c.lower() in
                           ("smiles", "smi", "structure", "canonical_smiles")), df.columns[0])
    if id_col is None:
        id_col = next((c for c in df.columns if c.lower() in
                       ("id", "name", "cid", "compound_id", "reagent_id", "idnumber")), None)
    df = df.rename(columns={smiles_col: "smiles"})
    if id_col:
        df = df.rename(columns={id_col: "id"})
    else:
        df["id"] = [f"R{i+1:03d}" for i in range(len(df))]
    df["id"] = df["id"].astype(str)
    df["mol"] = df["smiles"].astype(str).apply(to_mol)
    bad = int(df["mol"].isna().sum())
    if bad:
        print(f"  ⚠ dropped {bad} unparseable SMILES from {os.path.basename(str(path))}")
    return df[df["mol"].notna()].reset_index(drop=True)

print("Toolkit loaded ✓")

## 2 · Define your example reaction

Paste the **reagent(s)** and the **expected product** as plain SMILES.
Put **one** SMILES in `REAGENTS` for a 1-reactant reaction, **two** for a 2-reactant reaction.

*(The defaults show a primary-amine → 1,2,3,4-thiatriazole transformation. Replace them with your own.)*

In [ ]:
# ───────────────────────── EDIT ME ─────────────────────────
REAGENTS = ["CCN"]                 # 1 reagent  -> e.g. ["CCN"]
                                   # 2 reagents -> e.g. ["OC(=O)CC=O", "CC(=O)NC"]
PRODUCT  = "CCNC1=NN=NS1"          # the expected product SMILES
# ────────────────────────────────────────────────────────────

reagent_mols = [to_mol(s) for s in REAGENTS]
product_mol  = to_mol(PRODUCT)
assert all(m is not None for m in reagent_mols), "A reagent SMILES failed to parse."
assert product_mol is not None, "PRODUCT SMILES failed to parse."
N_REACTANTS = len(REAGENTS)
print(f"{N_REACTANTS}-reactant reaction")

legends = [f"Reagent {i+1}" for i in range(N_REACTANTS)] + ["Expected product"]
Draw.MolsToGridImage(reagent_mols + [product_mol], molsPerRow=N_REACTANTS + 1,
                     subImgSize=(300, 260), legends=legends)

## 3 · Generalize → build the template

Pick **one** of three routes to turn your single example into a reusable SMARTS/SMIRKS.
Whichever you run, it sets the variable **`RXN_SMARTS`**, which you can hand-edit afterwards.

- **A · Automatic (rxnmapper)** — no manual atom mapping needed; tune the `radius` for how much context to keep.
- **B · From a mapped example (rdchiral)** — you supply atom-map numbers (`:1`, `:2`, …) shared between reactants and product.
- **C · Manual** — just write the SMARTS yourself.

### A · Automatic — rxnmapper + rxnutils  *(handles unmapped SMILES)*

In [ ]:
def auto_template(reagent_mols, product_mol, radius=1):
    """rxnmapper atom-maps the example, then rxnutils extracts a forward template at `radius`."""
    if not HAVE_RXNMAPPER:
        raise RuntimeError("rxnmapper/rxnutils not installed — use route B or C.")
    reactants = ".".join(Chem.MolToSmiles(m) for m in reagent_mols)
    reaction  = f"{reactants}>>{Chem.MolToSmiles(product_mol)}"
    mapped = RXNMapper().get_attention_guided_atom_maps([reaction])[0]["mapped_rxn"]
    rxn = ChemicalReaction(mapped)
    tmpl = rxn.generate_reaction_template(radius=radius, expand_ring=False, expand_hetero=False)
    return tmpl[0].smarts

if HAVE_RXNMAPPER:
    RXN_SMARTS = auto_template(reagent_mols, product_mol, radius=1)
    show_template(RXN_SMARTS)
else:
    print("rxnmapper not installed — skip to route B or C.")

### B · From an atom-mapped example — rdchiral  *(lightweight)*

Add matching map numbers to the atoms that are conserved, e.g.
`[CH3:1][CH2:2][NH2:3]` → `[CH3:1][CH2:2][NH:3]C1=NN=NS1`.

In [ ]:
# Fill these in with atom-mapped SMILES, then run.
MAPPED_REACTANTS = ".".join(REAGENTS)     # e.g. "[CH3:1][CH2:2][NH2:3]"
MAPPED_PRODUCT   = PRODUCT                 # e.g. "[CH3:1][CH2:2][NH:3]C1=NN=NS1"

if not HAVE_RDCHIRAL:
    print("rdchiral not installed — use route A or C.")
elif ":" not in MAPPED_REACTANTS or ":" not in MAPPED_PRODUCT:
    print("No atom maps found. Add matching :1, :2, … labels to MAPPED_REACTANTS / MAPPED_PRODUCT,")
    print("then re-run this cell — or use route A (automatic) or route C (manual).")
else:
    res = extract_from_reaction(
        {"reactants": MAPPED_REACTANTS, "products": MAPPED_PRODUCT, "_id": "ex"}
    )
    if res and res.get("reaction_smarts"):
        RXN_SMARTS = ">>".join(res["reaction_smarts"].split(">>")[::-1])  # retro -> forward
        show_template(RXN_SMARTS)
    else:
        print("rdchiral could not extract a template — check that the same map numbers")
        print("appear on conserved atoms in both reactants and product.")

### C · Manual — write it yourself

In [ ]:
# Set RXN_SMARTS directly. Example single-reactant template (amine -> thiatriazole):
RXN_SMARTS = "[NX3;H2,H1;!$(NC=O):1]>>[N:1]C1=NN=NS1"

# Example two-reactant template (acid+aldehyde fragment + secondary amide -> imide):
# RXN_SMARTS = "[O]-[C:1](=[O:2])-[C:3]-[C:4]=[O:6].[C:8](=[O:9])-[NH1:10]>>[C:8](=[O:9])-[N:10]-[C:1](=[O:2])-[C:3]-[C:4]=[O:6]"

show_template(RXN_SMARTS)

## 4 · Validate the template

Apply `RXN_SMARTS` back to your **example reagents** and confirm it regenerates the **expected product**.
A green ✓ means the template is at least correct for the case you designed it on.

In [ ]:
result = validate_template(RXN_SMARTS, REAGENTS, PRODUCT)
if result["ok"]:
    print("✓ PASS — template reproduces the expected product.")
else:
    print("✗ FAIL — template did NOT reproduce the expected product.")
    print("   expected :", result.get("expected"))
    print("   produced :", result.get("produced") or "(no products)")
    print("   → tweak RXN_SMARTS (radius / atom environments) and re-run section 3–4.")

# Show what the template actually does to the example:
_rxn = rdChemReactions.ReactionFromSmarts(RXN_SMARTS)
_prods = run_template(_rxn, reagent_mols)
if _prods:
    show_reaction(reagent_mols, next(iter(_prods.values())), label="Template applied to your example:")

## 5 · Load reagent libraries (`.tsv`)

Point to your TSV file(s). The loader auto-detects a **SMILES** column and an **ID/name** column
(override with `smiles_col=` / `id_col=` if your headers are unusual).

- **1-reactant** reaction → set `LIB_A` only (leave `LIB_B = None`).
- **2-reactant** reaction → set both `LIB_A` and `LIB_B`.

A tiny demo TSV is written below so the notebook runs out-of-the-box — replace the paths with your files.

In [ ]:
# Demo files (delete once you point to your own .tsv libraries) ----------------
pd.DataFrame({"smiles": ["CCN", "c1ccccc1N", "NCC1CC1", "CC(C)N", "NCCO"],
              "id":     ["amine1","aniline","cpa","ipa","ethanolamine"]}
            ).to_csv("demo_libA.tsv", sep="\t", index=False)
# ------------------------------------------------------------------------------

LIB_A = "demo_libA.tsv"     # first reagent library  (required)
LIB_B = None                # second reagent library (None for 1-reactant reactions)

A = load_library(LIB_A)
print(f"Library A: {len(A)} reagents")
display(A[["id", "smiles"]].head())

if N_REACTANTS == 2:
    assert LIB_B is not None, "This is a 2-reactant reaction — set LIB_B."
    B = load_library(LIB_B)
    print(f"Library B: {len(B)} reagents")
    display(B[["id", "smiles"]].head())
else:
    B = None

## 6 · Enumerate

Apply the validated template across the librarie(s).

- 1-reactant: every reagent in **A**.
- 2-reactant: choose **`PAIRING`** = `"combinatorial"` (all A × B) or `"one_to_one"` (A[i] with B[i]).

`MAX_PAIRS` caps the run so a careless combinatorial explosion can't hang the notebook.
Products are deduplicated per pairing and tagged with MW / logP.

In [ ]:
PAIRING   = "combinatorial"     # "combinatorial" or "one_to_one"  (ignored for 1-reactant)
MAX_PAIRS = 2000                # safety cap on number of reactant combinations

rxn = rdChemReactions.ReactionFromSmarts(RXN_SMARTS)

# build the list of reactant combinations to try
if N_REACTANTS == 1:
    combos = [((row.id,), (row.mol,), (row.smiles,)) for row in A.itertuples()]
else:
    if PAIRING == "one_to_one":
        n = min(len(A), len(B))
        pairs = zip(A.head(n).itertuples(), B.head(n).itertuples())
    else:
        pairs = itertools.product(A.itertuples(), B.itertuples())
    combos = [((a.id, b.id), (a.mol, b.mol), (a.smiles, b.smiles)) for a, b in pairs]

if len(combos) > MAX_PAIRS:
    print(f"⚠ {len(combos)} combinations > MAX_PAIRS({MAX_PAIRS}); truncating.")
    combos = combos[:MAX_PAIRS]

rows = []
for ids, mols, smis in combos:
    for psmi, pmol in run_template(rxn, mols).items():
        mw, logp = props(pmol)
        rec = {"product_id": "_".join(ids) + "_P", "product_smiles": psmi,
               "MW": mw, "logP": logp, "_pmol": pmol, "_rmols": mols}
        for j, (rid, rsmi) in enumerate(zip(ids, smis), 1):
            rec[f"reagent{j}_id"] = rid
            rec[f"reagent{j}_smiles"] = rsmi
        rows.append(rec)

products = pd.DataFrame(rows)
print(f"Tried {len(combos)} combinations → {len(products)} products "
      f"({products['product_smiles'].nunique() if len(products) else 0} unique).")
display_cols = [c for c in ["product_id","reagent1_id","reagent2_id",
                            "product_smiles","MW","logP"] if c in products.columns]
display(products[display_cols].head(20))

## 7 · Visualize the 2D output

Render the enumerated reactions inline. `N_SHOW` limits how many are drawn (drawing is the slow part).

In [ ]:
from IPython.display import Image
N_SHOW = 12

if len(products) == 0:
    print("No products to display.")
else:
    for _, r in products.head(N_SHOW).iterrows():
        label = f"{r['product_id']}   MW={r['MW']}  logP={r['logP']}"
        display(Image(data=reaction_png(list(r["_rmols"]), r["_pmol"])))
        print(label, "\n")
    if len(products) > N_SHOW:
        print(f"... {len(products) - N_SHOW} more (raise N_SHOW to see them).")

A compact grid of just the **product** structures:

In [ ]:
if len(products):
    pmols = [r["_pmol"] for _, r in products.head(30).iterrows()]
    plegs = [f"{r['product_id']}\nMW {r['MW']}" for _, r in products.head(30).iterrows()]
    display(Draw.MolsToGridImage(pmols, molsPerRow=4, subImgSize=(260, 200), legends=plegs))

## 8 · Export *(optional)*

Save the results: a tidy **CSV**, an **Excel** workbook with embedded reaction images, and the individual
reaction **PNG**s. Set `OUTPUT_DIR` and run.

In [ ]:
OUTPUT_DIR = "rxn_output"
SAVE_CSV   = True
SAVE_XLSX  = True
SAVE_PNGS  = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
img_dir = os.path.join(OUTPUT_DIR, "reaction_images")
os.makedirs(img_dir, exist_ok=True)

if len(products) == 0:
    print("Nothing to export.")
else:
    tidy_cols = [c for c in products.columns if not c.startswith("_")]
    tidy = products[tidy_cols]

    if SAVE_CSV:
        p = os.path.join(OUTPUT_DIR, "products.csv")
        tidy.to_csv(p, index=False); print("CSV  ->", p)

    if SAVE_PNGS:
        for _, r in products.iterrows():
            with open(os.path.join(img_dir, f"{r['product_id']}.png"), "wb") as f:
                f.write(reaction_png(list(r["_rmols"]), r["_pmol"]))
        print("PNGs ->", img_dir)

    if SAVE_XLSX:
        from openpyxl import Workbook
        from openpyxl.drawing.image import Image as XLImage
        wb = Workbook(); ws = wb.active; ws.title = "products"
        headers = tidy_cols + ["reaction_2D"]
        for c, h in enumerate(headers, 1):
            ws.cell(row=1, column=c, value=h)
        for i, (_, r) in enumerate(products.iterrows(), start=2):
            for c, h in enumerate(tidy_cols, 1):
                ws.cell(row=i, column=c, value=r[h])
            png = reaction_png(list(r["_rmols"]), r["_pmol"], width=560, height=180)
            xl = XLImage(io.BytesIO(png)); xl.width, xl.height = 420, 135
            xl.anchor = ws.cell(row=i, column=len(headers)).coordinate
            ws.add_image(xl); ws.row_dimensions[i].height = 105
        for col, w in zip("ABCDEFG", [16, 16, 16, 38, 10, 10, 80]):
            ws.column_dimensions[col].width = w
        p = os.path.join(OUTPUT_DIR, "products.xlsx")
        wb.save(p); print("XLSX ->", p)

---
### Recap

You designed an example → generalized it into a **template** → **validated** it round-trips →
**enumerated** it across TSV libraries → **visualized** and **exported** the products.

To reuse for a new reaction, edit **section 2** (example), regenerate in **section 3**, and re-run downward.